# Attack Augmentation

Gera variantes adversariais de títulos de produtos (Abreviação, Notação de Unidades, Erro de Digitação) via LLM e exporta `data/attack.csv`.

**Saída:** `data/attack.csv` — consumida por `classifier_api.ipynb`.

**Passos:**
1. Ingestão e amostragem
2. Geração de variantes (batch LLM)


## Setup

In [ ]:
!pip install --upgrade pip
%pip install -q pandas numpy tqdm google-generativeai python-dotenv

In [ ]:
import os
import re
import json
import time
import logging
import pandas as pd
import numpy as np
from tqdm import tqdm
from pathlib import Path
from dotenv import load_dotenv
import google.generativeai as genai
import sys

load_dotenv()

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

In [ ]:
# ── Configuração ──────────────────────────────────────────────
INPUT_PATH   = "data/train.csv"       # CSV com títulos originais
INPUT_COLUMN = "title"
ATTACK_PATH  = "data/attack.csv"      # Saída deste notebook

RANDOM_SEED  = 42
BATCH_SIZE   = 5
API_KEY      = os.getenv("GOOGLE_API_KEY")
ATTACK_MODEL = "gemini-2.5-flash"


## Passo 1 — Ingestão e Amostragem


In [ ]:
df = pd.read_csv(INPUT_PATH)
print(f"Total de títulos: {len(df)}")
df.head(5)

In [ ]:
df_sample = df[[INPUT_COLUMN]].copy().reset_index(drop=True)
print(f"Títulos para processar: {len(df_sample)}")

In [ ]:
df_sample = df[[INPUT_COLUMN]].sample(n=100, random_state=RANDOM_SEED).reset_index(drop=True)
print(f"Títulos para processar: {len(df_sample)}")

In [ ]:
ATTACK_PROMPT = """
Você é um gerador de variantes adversariais de títulos de produtos de e-commerce brasileiro.

TAREFA
Para cada título recebido (identificado por idx), gere 6 (SEIS) variantes adversariais, distribuídas em 3 estratégias diferentes.
Retorne SOMENTE um array JSON. Sem texto extra, sem markdown.

O formato OBRIGATÓRIO do JSON para CADA objeto no array deve ser:
{
  "idx": int,
  "attacks": [
    {"tipo": "Abreviacao", "texto": "var1"},
    {"tipo": "Abreviacao", "texto": "var2"},
    {"tipo": "Notacao_Unidades", "texto": "var3"},
    {"tipo": "Notacao_Unidades", "texto": "var4"},
    {"tipo": "Erro_Digitacao", "texto": "var5"},
    {"tipo": "Erro_Digitacao", "texto": "var6"}
  ]
}

ESTRATÉGIAS OBRIGATÓRIAS (Faça exatamente 2 variantes diferentes para cada tipo):
  1. "Abreviacao": Abreviar palavras longas (ex: "Liquidificador" → "Liq.")
  2. "Notacao_Unidades": Notação alternativa de unidades (ex: "500ml" → "0,5L", "12kg" → "12 kilos")
  3. "Erro_Digitacao": Erro de digitação / Trocar caracteres adjacentes (ex: "Samsung" → "Samsugn")
"""

In [ ]:
def attack_batch_llm(
    titles: list[str],
    client: genai.GenerativeModel,
    start_idx: int = 0,
) -> list[dict]:
    lines = [f'[{start_idx + i}] "{t}"' for i, t in enumerate(titles)]
    user_prompt = "\n".join(lines)

    try:
        response = client.generate_content(user_prompt)
        raw_text = response.text.strip()
        if raw_text.startswith("```"):
            raw_text = re.sub(r"^```(?:json)?", "", raw_text).rstrip("`").strip()
        parsed = json.loads(raw_text)
        if isinstance(parsed, dict):
            parsed = [parsed]
        return parsed
    except json.JSONDecodeError:
        matches = re.findall(r"\{[^{}]+\}", raw_text, re.DOTALL)
        results = []
        for m in matches:
            try:
                results.append(json.loads(m))
            except json.JSONDecodeError:
                continue
        return results or [{"idx": start_idx + i, "attacked": t} for i, t in enumerate(titles)]
    except Exception as e:
        logger.error("Erro attack LLM: %s", e)
        return [{"idx": start_idx + i, "attacked": t} for i, t in enumerate(titles)]

In [ ]:
os.makedirs(os.path.dirname(ATTACK_PATH) or ".", exist_ok=True)

genai.configure(api_key=API_KEY)
attack_client = genai.GenerativeModel(
    model_name=ATTACK_MODEL,
    system_instruction=ATTACK_PROMPT,
    generation_config=genai.GenerationConfig(
        response_mime_type="application/json",
        temperature=0.7,  
        max_output_tokens=2048,
    ),
)

titles_orig = df_sample[INPUT_COLUMN].tolist()
n_attack = len(titles_orig)

registros_ataque = []

for batch_start in tqdm(range(0, n_attack, BATCH_SIZE), desc="Attack", unit="batch"):
    batch_end = min(batch_start + BATCH_SIZE, n_attack)
    batch = titles_orig[batch_start:batch_end]

    # Substitua pelo nome correto da sua função/cliente
    raw_list = attack_batch_llm(batch, attack_client, start_idx=batch_start)

    for local_i in range(batch_end - batch_start):
        global_i = batch_start + local_i
        
        # Procura a resposta correspondente deste idx do batch
        matched = next(
            (r for r in raw_list if isinstance(r, dict) and r.get("idx") == global_i),
            None,
        )
        
        # Confere se a IA gerou a lista de dicionários "attacks" corretamente
        if matched and "attacks" in matched and isinstance(matched["attacks"], list):
            ataques = matched["attacks"]
        else:
            # Fallback caso a IA falhe ou gagueje feio
            ataques = [{"tipo": "Fallback_Erro_LLM", "texto": titles_orig[global_i]}] * 6
            
        for ataque in ataques:
            # Pega o tipo e o texto gerado (com proteções caso a IA mande formato errado)
            if isinstance(ataque, dict):
                tipo = ataque.get("tipo", "Desconhecido")
                texto = ataque.get("texto", titles_orig[global_i])
            else:
                tipo = "Desconhecido"
                texto = str(ataque)

            registros_ataque.append({
                "id": global_i,
                "original": titles_orig[global_i],
                "tipo_variacao": tipo,
                "attacked": texto
            })

    time.sleep(2) # Pausa para não estourar limite da API

# Criando o DataFrame final
df_attack = pd.DataFrame(registros_ataque)
df_attack.to_csv(ATTACK_PATH, index=False, encoding="utf-8-sig")

print(f"Attack concluído: {len(titles_orig)} títulos geraram {len(df_attack)} linhas → {ATTACK_PATH}")
df_attack.head(15)